# UrbanEye AI - Phase 2: Custom Civic YOLOv8 Training

Trains a **custom YOLOv8n** detector on public civic datasets:
- `pothole` (Roboflow Universe: ~8,483 images)
- `garbage` (Roboflow Universe: ~5,980 images)

**Before running:**
1. Get a FREE Roboflow API key: https://app.roboflow.com/settings/api
2. Runtime -> Change runtime type -> **T4 GPU**
3. Paste your key in the config cell below, then *Run all*.

**Time:** ~60-120 min on free T4 depending on epochs. Output: `best.pt` -> drop into your backend.


In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

In [ ]:
# ================= CONFIG =================
RF_API_KEY = "PASTE_YOUR_FREE_ROBOFLOW_API_KEY_HERE"

# Public Roboflow Universe datasets (add more later: drainage, streetlight...)
DATASETS = [
    {"tag": "ph", "workspace": "yolov5-p91zm", "project": "pothole-r4q1p", "version": 1},
    {"tag": "gb", "workspace": "garbage-detection-oa9nh", "project": "yolov5-garbage-detection", "version": 1},
]

UNIFIED_CLASSES = ["garbage", "pothole"]
EPOCHS = 50          # 30 = faster (~75 min), 50 = better accuracy
IMGSZ = 640

assert not RF_API_KEY.startswith("PASTE"), "Get a free key: https://app.roboflow.com/settings/api"

In [ ]:
# ============ DOWNLOAD FROM ROBOFLOW ============
from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)
downloaded = []

for spec in DATASETS:
    ds = (rf.workspace(spec["workspace"])
            .project(spec["project"])
            .version(spec["version"])
            .download("yolov8"))
    downloaded.append({"spec": spec, "location": ds.location})
    print(f"[{spec['tag']}] ready at {ds.location}")

In [ ]:
# ===== MERGE INTO ONE UNIFIED DATASET =====
# Remaps each dataset's class ids onto UNIFIED_CLASSES, drops unrelated classes.
import glob, os, shutil, yaml

MERGED = "/content/merged"
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")

KEYWORD_TO_UNIFIED = {
    "poth": "pothole",
    "garb": "garbage",
    "trash": "garbage",
    "waste": "garbage",
    "litter": "garbage",
}

def unify_index(raw_name):
    n = str(raw_name).lower()
    for kw, unified in KEYWORD_TO_UNIFIED.items():
        if kw in n:
            return UNIFIED_CLASSES.index(unified)
    return None

def find_image(images_dir, stem):
    for ext in IMG_EXTS:
        p = os.path.join(images_dir, stem + ext)
        if os.path.exists(p):
            return p
    return None

for split in ("train", "valid"):
    os.makedirs(f"{MERGED}/images/{split}", exist_ok=True)
    os.makedirs(f"{MERGED}/labels/{split}", exist_ok=True)

stats = {c: 0 for c in UNIFIED_CLASSES}
skipped_boxes = 0

for entry in downloaded:
    tag = entry["spec"]["tag"]
    root = entry["location"]

    with open(os.path.join(root, "data.yaml")) as f:
        names = yaml.safe_load(f)["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]

    id_map = [unify_index(n) for n in names]
    print(f"[{tag}] source classes -> unified: {list(zip(names, id_map))}")

    for split in ("train", "valid", "test"):
        out_split = "train" if split == "train" else "valid"
        img_src = os.path.join(root, split, "images")
        lbl_src = os.path.join(root, split, "labels")
        if not os.path.isdir(lbl_src):
            continue

        for lbl_path in glob.glob(os.path.join(lbl_src, "*.txt")):
            stem = os.path.splitext(os.path.basename(lbl_path))[0]
            img_path = find_image(img_src, stem)
            if img_path is None:
                continue

            new_lines, used = [], set()
            for line in open(lbl_path):
                parts = line.split()
                if len(parts) != 5:
                    continue
                mapped = id_map[int(parts[0])]
                if mapped is None:
                    skipped_boxes += 1
                    continue
                new_lines.append(" ".join([str(mapped)] + parts[1:]))
                used.add(mapped)

            if not new_lines:
                continue

            new_stem = f"{tag}_{stem}"
            shutil.copy(img_path, f"{MERGED}/images/{out_split}/{new_stem}{os.path.splitext(img_path)[1]}")
            with open(f"{MERGED}/labels/{out_split}/{new_stem}.txt", "w") as f:
                f.write("\n".join(new_lines) + "\n")

            for c in used:
                stats[UNIFIED_CLASSES[c]] += 1

print("\nImages per class:", stats)
print("Boxes skipped (unmapped classes):", skipped_boxes)

In [ ]:
# ============ WRITE data.yaml + SANITY CHECK ============
data_yaml = {
    "path": MERGED,
    "train": "images/train",
    "val": "images/valid",
    "names": {i: c for i, c in enumerate(UNIFIED_CLASSES)},
}
with open(f"{MERGED}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

train_n = len(glob.glob(f"{MERGED}/images/train/*"))
val_n = len(glob.glob(f"{MERGED}/images/valid/*"))
print(f"data.yaml written | train={train_n}  valid={val_n}")
assert train_n > 200 and val_n > 40, "Too few images merged - check previous cell output."

In [ ]:
# ============ TRAIN YOLOv8n ============
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # COCO-pretrained nano, auto-downloads (~6MB)
model.train(
    data=f"{MERGED}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    name="civic_v1",
    patience=10,
)
BEST = "runs/detect/civic_v1/weights/best.pt"
print("\nBest weights saved at:", BEST)
# If Colab disconnects mid-training, resume with:
# YOLO("runs/detect/civic_v1/weights/last.pt").train(resume=True)

In [ ]:
# ============ VALIDATE METRICS ============
from IPython.display import Image as IPyImage, display

best = YOLO(BEST)
metrics = best.val(data=f"{MERGED}/data.yaml")
print(f"mAP@50    : {metrics.box.map50:.3f}")
print(f"mAP@50-95 : {metrics.box.map:.3f}")

cm = glob.glob("runs/detect/val*/confusion_matrix.png") + glob.glob("runs/detect/*/confusion_matrix.png")
if cm:
    display(IPyImage(cm[-1], width=480))

In [ ]:
# ============ VISUAL SANITY CHECK ============
samples = sorted(glob.glob(f"{MERGED}/images/valid/*"))[:4]
preds = best.predict(source=samples, conf=0.25, save=True)

for p in sorted(glob.glob(str(preds[0].save_dir) + "/*.jpg"))[:4]:
    display(IPyImage(p, width=420))

In [ ]:
# ============ EXPORT FOR LOCAL BACKEND ============
shutil.make_archive(
    "/content/civic_yolov8", "zip",
    root_dir=os.path.dirname(BEST),
    base_dir=os.path.basename(BEST),
)
from google.colab import files
files.download("/content/civic_yolov8.zip")
print("Download started -> extract best.pt from the zip.")

## Local integration (after download)

1. Extract `best.pt` from `civic_yolov8.zip`
2. Place it at: `UrbanEye/backend/ai/models/civic_yolov8.pt`
3. Restart backend (`backend/start.bat`)
4. Look for this log line on first upload:
   `[yolo_detector] Loaded CUSTOM civic_yolov8 ...`

**What changes:** uploads now get real bounding-box detection with severity computed from damaged-area ratio (resolution independent). Photos without civic issues fall through to the MobileNetV2 classifier automatically.

**Adding more classes later:** extend `UNIFIED_CLASSES` + add another dataset entry in the config cell (e.g., drainage / streetlight datasets), rerun from the download cell.